# 복약 지도 챗봇

In [1]:
SYSTEM_PROMPT = """
당신은 친절한 복약 안내 도우미입니다.

[역할]
- 환자가 약을 올바르게 복용할 수 있도록 쉽게 안내합니다.
- 주로 노인 환자를 대상으로 하므로 어렵고 복잡한 의학 용어를 절대 사용하지 않습니다.

[답변 규칙]
1. 문장은 짧고 간단하게 작성합니다.
2. 한 번에 한 가지 정보만 전달합니다.
3. 숫자는 "1정", "하루 2번"처럼 구체적으로 말합니다.
4. 반드시 "정확한 복용법은 담당 약사님께 확인하세요"를 마지막에 추가합니다.
5. 위험할 수 있는 질문(약 과다복용 등)은 즉시 병원/약국 방문을 권유합니다.

[금지 사항]
- 진단이나 처방은 절대 하지 않습니다.
- 확실하지 않은 정보는 말하지 않습니다.

[같이 먹으면 안 되는 주요 약물 조합]
아래 조합에 해당하는 질문을 받으면 반드시 위험성을 먼저 안내하세요.

1. 해열진통제 + 종합감기약
   - 종합감기약에는 이미 해열진통제 성분(아세트아미노펜 등)이 포함된 경우가 많습니다.
   - 함께 복용하면 같은 성분이 두 배로 들어가 간 손상이나 위장 출혈 위험이 커집니다.

2. 서로 다른 소염진통제 중복 복용
   - 아스피린, 이부프로펜, 나프록센 등 비스테로이드성 소염진통제(NSAIDs)는 원리가 같습니다.
   - 중복 복용하면 약효는 늘지 않고 부작용 위험만 높아집니다.

3. 항생제 + 제산제
   - 제산제(위산 중화제)는 항생제의 흡수를 방해하여 약효를 떨어뜨립니다.
   - 항생제 복용 후 최소 2시간 간격을 두고 제산제를 드세요.

4. 고지혈증 치료제 + 항진균제
   - 특정 성분들이 충돌하여 근육 손상 등의 부작용이 생길 수 있습니다.
   - 두 약을 함께 처방받은 경우 반드시 의사·약사에게 확인하세요.

5. 당뇨약 + 스테로이드 또는 이뇨제
   - 스테로이드(부신피질호르몬제)나 일부 이뇨제는 혈당을 올릴 수 있습니다.
   - 당뇨약의 효과가 약해져 혈당 조절이 어려워질 수 있습니다.

6. 임산부가 피해야 할 약물
   - 피해야 할 약물 (특히 임신 3기): 이부프로펜, 나프록센, 덱시부프로펜 등 소염진통제(NSAIDs).
   - 주의가 필요한 성분: 슈도에페드린(코막힘), 페닐레프린.
   - 비교적 안전한 약물: 아세트아미노펜(타이레놀) 계열, 세파 계열 항생제 등.

"""

In [ ]:
import requests
import datetime
import dotenv
import os
import re
import base64
import gradio as gr
dotenv.load_dotenv()
AZURE_SPEECH_KEY = os.getenv('AZURE_SPEECH_KEY')

#====================== STT ==========================
def request_stt(audio_path):
    endpoint = "https://eastus.stt.speech.microsoft.com/speech/recognition/conversation/cognitiveservices/v1?language=ko-KR&format=detailed"
    headers = {
        "Ocp-Apim-Subscription-Key": AZURE_SPEECH_KEY ,
        "Content-Type": "audio/wav"
    }
    with open(audio_path, 'rb') as f:
        audio_data = f.read()
    response = requests.post(endpoint, headers=headers, data=audio_data)
    if not response.ok:
        return None
    return response.json()['NBest'][0]['Display']

#====================== TTS ==========================
def request_tts(input_text):
    endpoint = "https://eastus.tts.speech.microsoft.com/cognitiveservices/v1"
    headers = {
        "Ocp-Apim-Subscription-Key": AZURE_SPEECH_KEY,
        "Content-Type": "application/ssml+xml",
        "X-Microsoft-OutputFormat": "riff-8khz-16bit-mono-pcm"
    }
    body = f"""<speak version='1.0' xml:lang='en-US'>
        <voice xml:lang='ko-KR' xml:gender='Female' name='ko-KR-SunHi:DragonHDLatestNeural'>
            {input_text}
        </voice>
    </speak>"""
    response = requests.post(endpoint, headers=headers, data=body)
    if not response.ok:
        return None
    now = datetime.datetime.now()
    file_name = "tts_{}.wav".format(now.strftime("%Y%m%d_%H%M%S"))
    with open(file_name, "wb") as f:
        f.write(response.content)
    return file_name

#====================== AI (텍스트) ==========================
def request_openai(prompt, histories):
    dotenv.load_dotenv()
    KEY = os.getenv("OPEN_AI_KEY2")
    endpoint = "https://fimtrus-foundry.cognitiveservices.azure.com/openai/deployments/gpt-4.1/chat/completions?api-version=2025-01-01-preview"
    headers = {"api-key": KEY, "Content-Type": "application/json"}
    message_list = [{"role": "system", "content": SYSTEM_PROMPT}]
    for h in histories:
        message_list.append({"role": h['role'], "content": h['content']})
    message_list.append({"role": "user", "content": prompt})
    body = {"messages": message_list, "max_tokens": 4096, "temperature": 0.7, "top_p": 0.95}
    response = requests.post(endpoint, headers=headers, json=body)
    rj = response.json()
    content = rj['choices'][0]['message']['content']
    role = rj['choices'][0]['message']['role']
    return {"role": role, "content": content}

#====================== AI (Vision) ==========================
def request_openai_vision(image_path, prompt, histories):
    dotenv.load_dotenv()
    KEY = os.getenv("OPEN_AI_KEY2")
    endpoint = "https://fimtrus-foundry.cognitiveservices.azure.com/openai/deployments/gpt-4.1/chat/completions?api-version=2025-01-01-preview"
    headers = {"api-key": KEY, "Content-Type": "application/json"}

    ext = os.path.splitext(image_path)[1].lower().lstrip(".")
    mime = "jpeg" if ext in ("jpg", "jpeg") else ext
    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")

    message_list = [{"role": "system", "content": SYSTEM_PROMPT}]
    for h in histories:
        message_list.append({"role": h['role'], "content": h['content']})
    message_list.append({
        "role": "user",
        "content": [
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": f"data:image/{mime};base64,{b64}"}}
        ]
    })
    body = {"messages": message_list, "max_tokens": 4096, "temperature": 0.7, "top_p": 0.95}
    response = requests.post(endpoint, headers=headers, json=body)
    rj = response.json()
    content = rj['choices'][0]['message']['content']
    role = rj['choices'][0]['message']['role']
    return {"role": role, "content": content}

#====================== 공통 처리 ==========================
def histories_to_display(histories):
    """히스토리 → Chatbot messages(dict) 형식 변환"""
    return [{"role": h['role'], "content": h['content']} for h in histories]

def process_question(input_text, histories):
    ai_data = request_openai(input_text, histories)
    new_histories = histories + [
        {"role": "user", "content": input_text},
        ai_data
    ]
    cleaned = re.sub(r"[^가-힣a-zA-Z0-9\s!.,]", "", ai_data['content'])
    tts_file = request_tts(cleaned)
    return histories_to_display(new_histories), new_histories, tts_file

def process_image(image_path, prompt, histories):
    ai_data = request_openai_vision(image_path, prompt, histories)
    new_histories = histories + [
        {"role": "user", "content": f"[약 사진 분석 요청] {prompt}"},
        ai_data
    ]
    cleaned = re.sub(r"[^가-힣a-zA-Z0-9\s!.,]", "", ai_data['content'])
    tts_file = request_tts(cleaned)
    return histories_to_display(new_histories), new_histories, tts_file

#======================화면 구성=======================
with gr.Blocks(title="💊 AI 복약 안내 도우미") as demo:
    gr.Markdown("# 💊 AI 복약 안내 도우미\n### 약에 대해 궁금한 점을 음성이나 글로 물어보세요")

    chat_history = gr.State([])

    with gr.Column():
        chatbot = gr.Chatbot(scale=10, label="복약 지도 챗봇")

        with gr.Row():
            with gr.Column():
                gr.Markdown("### 🎤 말하기")
                audio_input = gr.Audio(sources="microphone", type="filepath", label="마이크")
                gr.Markdown("#### 📁 오디오 파일 업로드")
                audio_upload = gr.Audio(sources="upload", type="filepath", label="오디오 파일 선택")
                audio_upload_btn = gr.Button("📤 업로드 파일 전송", variant="secondary")

            with gr.Column():
                gr.Markdown("### ⌨️ 글로 묻기")
                text_input = gr.Textbox(placeholder="예: 혈압약 언제 먹어야 하나요?", label="질문", lines=2)
                send_btn = gr.Button("💬 보내기", variant="primary")

            with gr.Column():
                gr.Markdown("### 📷 약 사진 분석")
                image_input = gr.Image(type="filepath", label="약 봉투 / 약통 사진 업로드")
                image_prompt = gr.Textbox(
                    value="이 약의 복용법과 주의사항을 알려주세요.",
                    label="질문 (선택)",
                    lines=2
                )
                image_btn = gr.Button("🔍 사진 분석하기", variant="secondary")

        audio_output = gr.Audio(label="AI 음성 답변", autoplay=True)

    # ---- 이벤트 함수 ----
    def on_mic_stop(audio_path, histories):
        """녹음 중지 시 STT → OpenAI → TTS 자동 처리"""
        if not audio_path:
            return histories_to_display(histories), histories, None, ""
        text = request_stt(audio_path)
        if not text:
            return histories_to_display(histories), histories, None, "STT 인식 실패"
        display, new_histories, tts_file = process_question(text, histories)
        return display, new_histories, tts_file, text

    def on_audio_upload_send(audio_path, histories):
        """업로드된 오디오 파일을 STT → OpenAI → TTS로 처리"""
        if not audio_path:
            return histories_to_display(histories), histories, None, ""
        text = request_stt(audio_path)
        if not text:
            return histories_to_display(histories), histories, None, "STT 인식 실패"
        display, new_histories, tts_file = process_question(text, histories)
        return display, new_histories, tts_file, text

    def on_send_click(input_text, histories):
        if not input_text or not input_text.strip():
            return histories_to_display(histories), histories, None
        display, new_histories, tts_file = process_question(input_text, histories)
        return display, new_histories, tts_file

    def on_image_analyze(image_path, prompt, histories):
        if not image_path:
            return histories_to_display(histories), histories, None
        if not prompt or not prompt.strip():
            prompt = "이 약의 복용법과 주의사항을 알려주세요."
        display, new_histories, tts_file = process_image(image_path, prompt, histories)
        return display, new_histories, tts_file

    # ---- 이벤트 연결 ----
    # 녹음 중지 시 자동으로 STT → AI → TTS 처리
    audio_input.stop_recording(
        on_mic_stop,
        inputs=[audio_input, chat_history],
        outputs=[chatbot, chat_history, audio_output, text_input]
    )

    # 오디오 업로드: 버튼 클릭으로 전송
    audio_upload_btn.click(
        on_audio_upload_send,
        inputs=[audio_upload, chat_history],
        outputs=[chatbot, chat_history, audio_output, text_input]
    )

    send_btn.click(on_send_click, inputs=[text_input, chat_history], outputs=[chatbot, chat_history, audio_output])
    image_btn.click(
        on_image_analyze,
        inputs=[image_input, image_prompt, chat_history],
        outputs=[chatbot, chat_history, audio_output]
    )

demo.launch(theme=gr.themes.Soft())


SyntaxError: expression expected after dictionary key and ':' (3554081129.py, line 15)

In [ ]:
import requests
import datetime
import dotenv
import os
import re
import base64
import gradio as gr

dotenv.load_dotenv()
OPEN_AI_KEY2 = os.getenv('OPEN_AI_KEY2')
AZURE_SPEECH_KEY = os.getenv('AZURE_SPEECH_KEY')

#====================== STT ==========================
def request_stt(audio_path):
    endpoint = "https://eastus.stt.speech.microsoft.com/speech/recognition/conversation/cognitiveservices/v1?language=ko-KR&format=detailed"
    headers = {
        "Ocp-Apim-Subscription-Key": AZURE_SPEECH_KEY,
        "Content-Type": "audio/wav"
    }
    with open(audio_path, 'rb') as f:
        audio_data = f.read()
    response = requests.post(endpoint, headers=headers, data=audio_data)
    if not response.ok:
        return None
    data = response.json()
    if data.get('RecognitionStatus') != 'Success' or not data.get('NBest'):
        return None
    return data['NBest'][0]['Display']

#====================== TTS ==========================
def request_tts(input_text):
    endpoint = "https://eastus.tts.speech.microsoft.com/cognitiveservices/v1"
    headers = {
        "Ocp-Apim-Subscription-Key": AZURE_SPEECH_KEY,
        "Content-Type": "application/ssml+xml",
        "X-Microsoft-OutputFormat": "riff-8khz-16bit-mono-pcm"
    }
    body = f"""<speak version='1.0' xml:lang='en-US'>
        <voice xml:lang='ko-KR' xml:gender='Female' name='ko-KR-SunHi:DragonHDLatestNeural'>
            {input_text}
        </voice>
    </speak>"""
    response = requests.post(endpoint, headers=headers, data=body)
    if not response.ok:
        return None
    now = datetime.datetime.now()
    file_name = "tts_{}.wav".format(now.strftime("%Y%m%d_%H%M%S"))
    with open(file_name, "wb") as f:
        f.write(response.content)
    return file_name

#====================== AI (텍스트) ==========================
def request_openai(prompt, histories):
    dotenv.load_dotenv()
    KEY = os.getenv("OPEN_AI_KEY2")
    endpoint = "https://fimtrus-foundry.cognitiveservices.azure.com/openai/deployments/gpt-4.1/chat/completions?api-version=2025-01-01-preview"
    headers = {"api-key": KEY, "Content-Type": "application/json"}
    message_list = [{"role": "system", "content": SYSTEM_PROMPT}]
    for h in histories:
        message_list.append({"role": h["role"], "content": h["content"]})
    message_list.append({"role": "user", "content": prompt})
    body = {"messages": message_list, "max_tokens": 4096, "temperature": 0.7, "top_p": 0.95}
    response = requests.post(endpoint, headers=headers, json=body)
    rj = response.json()
    content = rj['choices'][0]['message']['content']
    role = rj['choices'][0]['message']['role']
    return {"role": role, "content": content}

#====================== AI (Vision) ==========================
def request_openai_vision(image_path, prompt, histories):
    dotenv.load_dotenv()
    KEY = os.getenv("OPEN_AI_KEY2")
    endpoint = "https://fimtrus-foundry.cognitiveservices.azure.com/openai/deployments/gpt-4.1/chat/completions?api-version=2025-01-01-preview"
    headers = {"api-key": KEY, "Content-Type": "application/json"}
    ext = os.path.splitext(image_path)[1].lower().lstrip(".")
    mime = "jpeg" if ext in ("jpg", "jpeg") else ext
    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    message_list = [{"role": "system", "content": SYSTEM_PROMPT}]
    for h in histories:
        message_list.append({"role": h["role"], "content": h["content"]})
    message_list.append({
        "role": "user",
        "content": [
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": f"data:image/{mime};base64,{b64}"}}
        ]
    })
    body = {"messages": message_list, "max_tokens": 4096, "temperature": 0.7, "top_p": 0.95}
    response = requests.post(endpoint, headers=headers, json=body)
    rj = response.json()
    content = rj['choices'][0]['message']['content']
    role = rj['choices'][0]['message']['role']
    return {"role": role, "content": content}

#====================== 공통 처리 ==========================
def to_display(histories):
    """내부 histories → Chatbot 표시용 dict 리스트"""
    return [{"role": h["role"], "content": h["content"]} for h in histories]

def process_question(input_text, histories):
    ai_data = request_openai(input_text, histories)
    new_histories = list(histories) + [
        {"role": "user", "content": input_text},
        {"role": ai_data["role"], "content": ai_data["content"]}
    ]
    cleaned = re.sub(r"[^가-힣a-zA-Z0-9\s!.,]", "", ai_data["content"])
    tts_file = request_tts(cleaned)
    return to_display(new_histories), new_histories, tts_file

def process_image(image_path, prompt, histories):
    ai_data = request_openai_vision(image_path, prompt, histories)
    new_histories = list(histories) + [
        {"role": "user", "content": f"[약 사진 분석 요청] {prompt}"},
        {"role": ai_data["role"], "content": ai_data["content"]}
    ]
    cleaned = re.sub(r"[^가-힣a-zA-Z0-9\s!.,]", "", ai_data["content"])
    tts_file = request_tts(cleaned)
    return to_display(new_histories), new_histories, tts_file

#======================화면 구성=======================
with gr.Blocks(title="💊 AI 복약 안내 도우미 v2") as demo2:
    gr.Markdown("# 💊 AI 복약 안내 도우미\n### 약에 대해 궁금한 점을 음성이나 글로 물어보세요")

    chat_history = gr.State([])

    with gr.Column():
        # type 파라미터 없이 사용 (이 버전 Gradio는 기본이 messages 형식)
        chatbot = gr.Chatbot(scale=10, label="복약 지도 챗봇")

        with gr.Row():
            with gr.Column():
                gr.Markdown("### 🎤 말하기")
                audio_input = gr.Audio(sources="microphone", type="filepath", label="마이크")
                gr.Markdown("#### 📁 오디오 파일 업로드")
                audio_upload = gr.Audio(sources="upload", type="filepath", label="오디오 파일 선택")
                audio_upload_btn = gr.Button("📤 업로드 파일 전송", variant="secondary")

            with gr.Column():
                gr.Markdown("### ⌨️ 글로 묻기")
                text_input = gr.Textbox(placeholder="예: 혈압약 언제 먹어야 하나요?", label="질문", lines=2)
                send_btn = gr.Button("💬 보내기", variant="primary")

            with gr.Column():
                gr.Markdown("### 📷 약 사진 분석")
                image_input = gr.Image(type="filepath", label="약 봉투 / 약통 사진 업로드")
                image_prompt = gr.Textbox(
                    value="이 약의 복용법과 주의사항을 알려주세요.",
                    label="질문 (선택)",
                    lines=2
                )
                image_btn = gr.Button("🔍 사진 분석하기", variant="secondary")

        audio_output = gr.Audio(label="AI 음성 답변", autoplay=True)

    # ---- 이벤트 함수 ----
    def on_mic_stop(audio_path, histories):
        if not audio_path:
            return to_display(histories), histories, None, ""
        text = request_stt(audio_path)
        if not text:
            return to_display(histories), histories, None, "STT 인식 실패"
        display, new_histories, tts_file = process_question(text, histories)
        return display, new_histories, tts_file, text

    def on_audio_upload_send(audio_path, histories):
        if not audio_path:
            return to_display(histories), histories, None, ""
        text = request_stt(audio_path)
        if not text:
            return to_display(histories), histories, None, "STT 인식 실패"
        display, new_histories, tts_file = process_question(text, histories)
        return display, new_histories, tts_file, text

    def on_send_click(input_text, histories):
        if not input_text or not input_text.strip():
            return to_display(histories), histories, None
        display, new_histories, tts_file = process_question(input_text, histories)
        return display, new_histories, tts_file

    def on_image_analyze(image_path, prompt, histories):
        if not image_path:
            return to_display(histories), histories, None
        if not prompt or not prompt.strip():
            prompt = "이 약의 복용법과 주의사항을 알려주세요."
        display, new_histories, tts_file = process_image(image_path, prompt, histories)
        return display, new_histories, tts_file

    # ---- 이벤트 연결 ----
    audio_input.stop_recording(
        on_mic_stop,
        inputs=[audio_input, chat_history],
        outputs=[chatbot, chat_history, audio_output, text_input]
    )
    audio_upload_btn.click(
        on_audio_upload_send,
        inputs=[audio_upload, chat_history],
        outputs=[chatbot, chat_history, audio_output, text_input]
    )
    send_btn.click(
        on_send_click,
        inputs=[text_input, chat_history],
        outputs=[chatbot, chat_history, audio_output]
    )
    image_btn.click(
        on_image_analyze,
        inputs=[image_input, image_prompt, chat_history],
        outputs=[chatbot, chat_history, audio_output]
    )

demo2.launch(theme=gr.themes.Soft(), share=True)


* Running on local URL:  http://127.0.0.1:7867
* Running on public URL: https://6b709833f518470661.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\gradio\queueing.py", line 766, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\gradio\route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\gradio\blocks.py", line 2158, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\gradio\blocks.py", line 1634, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\USER\AppData\Local\Progra